In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, RepeatVector
from tensorflow.keras.models import Model

# 1. Dataset
sentences = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# 2. Tokenization and padding
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
max_len = max(len(seq) for seq in sequences)
vocab_size = len(tokenizer.word_index) + 1

X = pad_sequences(
    sequences, maxlen=max_len, padding="post"
)

print("Vocabulary size:", vocab_size)
print("Padded sequences:\n", X)

# 3. Build the VAE encoder
embedding_dim = 16
latent_dim = 4

encoder_input = Input(shape=(max_len,))
embedded = Embedding(vocab_size, embedding_dim)(encoder_input)
encoded = LSTM(16)(embedded)

z_mean = Dense(latent_dim, name="z_mean")(encoded)
z_log_var = Dense(latent_dim, name="z_log_var")(encoded)

def sampling(args):
    mean, log_var = args
    epsilon = tf.random.normal(shape=tf.shape(mean))
    return mean + tf.exp(0.5 * log_var) * epsilon

z = tf.keras.layers.Lambda(sampling)([z_mean, z_log_var])

encoder = Model(
    encoder_input, [z_mean, z_log_var, z], name="encoder"
)

# 4. Build the decoder
latent_input = Input(shape=(latent_dim,))
repeated = RepeatVector(max_len)(latent_input)
decoded = LSTM(16, return_sequences=True)(repeated)
decoder_output = Dense(vocab_size, activation="softmax")(decoded)

decoder = Model(latent_input, decoder_output, name="decoder")

# 5. Train the VAE
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

@tf.function
def train_step(x):
    with tf.GradientTape() as tape:
        mean, log_var, z_sample = encoder(x, training=True)
        predictions = decoder(z_sample, training=True)

        # Reconstruction loss; ignore padding tokens
        token_loss = tf.keras.losses.sparse_categorical_crossentropy(
            x, predictions
        )
        mask = tf.cast(tf.not_equal(x, 0), tf.float32)
        reconstruction_loss = tf.reduce_sum(
            token_loss * mask
        ) / tf.reduce_sum(mask)

        # KL divergence regularization
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1 + log_var - tf.square(mean) - tf.exp(log_var),
                axis=1
            )
        )

        total_loss = reconstruction_loss + 0.01 * kl_loss

    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(total_loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return total_loss

# Train for 300 epochs
for epoch in range(300):
    loss = train_step(tf.constant(X, dtype=tf.int32))
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.numpy():.4f}")

# 6. Reconstruct the original sentences
print("\nOriginal and reconstructed sentences:")

mean, log_var, _ = encoder.predict(X, verbose=0)
predictions = decoder.predict(mean, verbose=0)
word_index = tokenizer.index_word

for i, sentence in enumerate(sentences):
    predicted_ids = np.argmax(predictions[i], axis=-1)
    words = [
        word_index.get(int(idx), "")
        for idx in predicted_ids
        if idx != 0
    ]
    reconstructed = " ".join(words)

    print("Original:", sentence)
    print("Reconstructed:", reconstructed)
    print()


Vocabulary size: 14
Padded sequences:
 [[ 1  3  2  4]
 [ 1  5  2  6]
 [ 1  7  2  8]
 [ 1  9  2 10]
 [ 1 11 12 13]]
Epoch 50, Loss: 1.3684
Epoch 100, Loss: 0.5904
Epoch 150, Loss: 0.3358
Epoch 200, Loss: 0.2437
Epoch 250, Loss: 0.1486
Epoch 300, Loss: 0.0953

Original and reconstructed sentences:
Original: The sky is blue
Reconstructed: the sky is blue

Original: The sun is bright
Reconstructed: the sun is bright

Original: The grass is green
Reconstructed: the grass is green

Original: The night is dark
Reconstructed: the night is dark

Original: The stars are shining
Reconstructed: the stars are shining

